# Notebook 6 — Toward a Tiny GPT

이제 남은 모듈들을 하나씩 추가하여 **좀 더 GPT다운 구조**로 갑니다.

이번 노트북에서 추가하는 것들:

- multi-head attention
- feedforward network
- residual connection
- layer normalization
- block stacking

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# ---------------------------------------------------------------------------
# [기본 데이터 전처리 및 로드]
# 이 부분은 지난 노트북과 동일합니다. 텍스트 데이터를 읽어와 문자를 정수로 인덱싱합니다.
# ---------------------------------------------------------------------------
if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

# 문맥의 길이(block_size)를 32에서 64로 확장했습니다. 모델이 한 번에 더 긴 문장을 봅니다.
block_size = 64
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

## 1. Multi-head attention

In [2]:
"""## 1. Multi-head attention"""

class Head(nn.Module):
    """
    하나의 어텐션 '머리(Head)'를 의미합니다.
    문장 속 단어들 간의 연관성(가중치)을 계산하는 단일 채널입니다.
    """
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        # 선형 회귀의 가중치 행렬을 정의하듯, Q, K, V를 위한 선형 변환을 정의합니다.
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)

        # 미래의 토큰을 보지 못하게 가려주는 하삼각행렬 마스크입니다.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

        # [드롭아웃(Dropout)]: 학습 시 무작위로 일부 가중치 연결을 0으로 만들어 끊어버립니다.
        # 통계학에서 변수가 특정 데이터에 너무 과도하게 적합(Overfitting)되는 것을 막는 일종의 규제(Regularization) 기법입니다.
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        # 단어 간의 유사도 점수를 계산합니다. (내적 연산)
        # k.size(-1)은 head_size를 의미하며, 이 값의 제곱근으로 나누어 정규화(Scaling)합니다.
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)

        # 미래 시점의 점수를 -무한대로 밀어내어 차단합니다.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # Softmax를 통해 각 행의 합이 1이 되는 확률 분포(어텐션 가중치)로 변환합니다.
        wei = F.softmax(wei, dim=-1)

        # 앞서 정의한 드롭아웃을 가중치 행렬에 적용하여 무작위로 일부 연결을 숨깁니다.
        wei = self.dropout(wei)

        # 최종적으로 가중치가 반영된 Value 정보를 추출합니다.
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """
    위에서 만든 'Head'를 여러 개 병렬로 수행하는 클래스입니다.
    데이터 분석에서 하나의 데이터셋을 여러 가지 파생 변수나 다른 관점으로 동시에 분석하는 것과 같습니다.
    """
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        # 예: 전체 차원(emb_dim)이 128이고 머리 개수(num_heads)가 4라면, 각 머리는 32차원(head_size)씩 담당합니다.
        head_size = emb_dim // num_heads

        # 파이썬 리스트처럼 여러 개의 모델 층을 담아두는 nn.ModuleList를 사용해 병렬 머리들을 생성합니다.
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])

        # 여러 머리들이 각자 분석한 결과물들을 다시 하나의 표(Matrix)로 병합한 후, 차원을 맞춰주기 위한 선형 변환 층입니다.
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # [torch.cat]: 각 머리가 계산한 결과(B, T, head_size)들을 마지막 차원(dim=-1)을 기준으로 가로로 이어 붙입니다.
        # Pandas에서 pd.concat([df1, df2], axis=1)로 컬럼을 합치는 과정과 완전히 동일합니다.
        out = torch.cat([h(x) for h in self.heads], dim=-1) # 합쳐진 결과의 차원: (B, T, emb_dim)

        # 합쳐진 특징들을 다시 한번 선형 결합(Projection)하여 최적의 형태로 조합합니다.
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 2. Feedforward + Block

In [3]:
"""## 2. Feedforward + Block"""

class FeedForward(nn.Module):
    """
    어텐션이 '단어 간의 관계'를 파악했다면, 이 층은 '각 단어 자체의 의미'를 더 깊게 수치화하는 선형 신경망입니다.
    회귀 분석 모형을 연속적으로 쌓은 인공신경망(MLP) 구조입니다.
    """
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        # nn.Sequential은 여러 연산 단계를 하나의 파이프라인으로 묶어 줍니다. (scikit-learn의 Pipeline과 유사)
        self.net = nn.Sequential(
            # 1단계: 차원을 4배로 뻥튀기하여 비선형 공간에서 더 많은 특징을 잡아낼 수 있도록 펼쳐줍니다. (128 -> 512)
            nn.Linear(emb_dim, 4 * emb_dim),
            # 2단계: 활성화 함수 ReLU를 적용합니다. 0보다 작은 값은 0으로 만들고, 0보다 큰 값은 그대로 둡니다.
            # 데이터 분석에서 비선형 왜곡 관계를 모형에 반영하기 위해 로그(log)나 제곱근을 취해주는 것과 비슷한 맥락입니다.
            nn.ReLU(),
            # 3단계: 확장했던 차원을 다시 원래의 차원으로 복원합니다. (512 -> 128)
            nn.Linear(4 * emb_dim, emb_dim),
            # 4단계: 과적합 방지를 위한 드롭아웃
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Transformer/GPT의 가장 기본이 되는 '기둥 벽돌(Block)'입니다.
    이 블록 안에서 [LayerNorm -> Attention -> 잔차 연결] 구조가 수행됩니다.
    """
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        # [LayerNorm (레이어 정규화)]: 데이터 분석 시 데이터의 스케일이 다르면 분석이 꼬이므로
        # StandardScaler를 통해 평균 0, 분산 1로 만드는 스케일링 작업을 하듯이, 신경망 내부 데이터의 단위를 맞춰주는 역할을 합니다.
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        # [잔차 연결 (Residual Connection / Skip Connection)]: 'x + f(x)' 구조입니다.
        # 원본 데이터인 x를 옆의 길로 그대로 전달해서, 연산을 거친 값에 더해줍니다.
        # 시계열 분석이나 회귀 분석에서 기존의 추세(Trend)를 그대로 유지한 채 '오차(Residual)' 혹은 '추가 변동분'만 모델이 학습하도록 유도하는 방식입니다.
        # 층이 깊어져도 데이터 고유의 신호가 희석되지 않고 끝까지 전달되도록 돕습니다.
        x = x + self.sa(self.ln1(x))   # 정규화한 데이터를 어텐션에 통과시키고, 그 결과를 원본 x에 더함
        x = x + self.ffwd(self.ln2(x)) # 다시 정규화하여 피드포워드에 통과시키고, 그 결과를 더함
        return x


"""## 3. Tiny GPT"""

class TinyGPT(nn.Module):
    """
    위에서 설계한 블록들을 수직으로 층층이 쌓아 올린 최종 GPT 모델 객체입니다.
    """
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        # 문자의 고유 인덱스를 밀집 벡터(실수 배열)로 매핑하는 임베딩 층
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        # 위치 정보를 벡터로 매핑하는 임베딩 층
        self.position_embedding = nn.Embedding(block_size, emb_dim)

        # [Block Stack]: Block을 num_layers(4번)만큼 반복해서 쌓아 올립니다.
        # 리스트 컴프리헨션 앞에 붙은 아스태리스크(*) 기호는 리스트 내의 원소들을 낱개로 풀어서(Unpacking) nn.Sequential에 순서대로 주입하라는 뜻입니다.
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])

        # 모든 블록을 거쳐 나온 최종 데이터를 마지막으로 한 번 더 정규화(스케일링)해 줍니다.
        self.ln_f = nn.LayerNorm(emb_dim)

        # 마지막으로 128차원의 데이터를 단어장 크기(vocab_size)로 복원하여 각 글자가 나타날 확률 점수(Logits)를 출력합니다.
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device) # 0부터 T-1까지의 정수 배열 생성

        tok = self.token_embedding(x) # 단어의 의미적 임베딩 결과 (B, T, emb_dim)
        pos = self.position_embedding(pos)[None] # 위치 임베딩 결과 (1, T, emb_dim) -> 브로드캐스팅되어 B행만큼 확장됨

        # 통계학의 가법 모델(Additive Model)처럼, 단어 의미 벡터에 위치 벡터를 더해 최종 입력값을 만듭니다.
        h = tok + pos

        h = self.blocks(h) # 4개의 GPT 블록을 순차적으로 통과시킵니다.
        h = self.ln_f(h)   # 최종 레이어 정규화
        logits = self.lm_head(h) # 선형 변환을 거쳐 예측 점수 도출
        return logits

# 모델 인스턴스를 생성하고 임의의 입력 데이터(xb)를 주입해 봅니다.
model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape) # 출력 결과: (64, 64, 65) -> (Batch_size, Sequence_length, Vocab_size)

logits.shape: torch.Size([64, 64, 65])


## 3. Tiny GPT

In [4]:
"""## 3. Tiny GPT"""

class TinyGPT(nn.Module):
    """
    위에서 설계한 블록들을 수직으로 층층이 쌓아 올린 최종 GPT 모델 객체입니다.
    """
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        # 문자의 고유 인덱스를 밀집 벡터(실수 배열)로 매핑하는 임베딩 층
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        # 위치 정보를 벡터로 매핑하는 임베딩 층
        self.position_embedding = nn.Embedding(block_size, emb_dim)

        # [Block Stack]: Block을 num_layers(4번)만큼 반복해서 쌓아 올립니다.
        # 리스트 컴프리헨션 앞에 붙은 아스태리스크(*) 기호는 리스트 내의 원소들을 낱개로 풀어서(Unpacking) nn.Sequential에 순서대로 주입하라는 뜻입니다.
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])

        # 모든 블록을 거쳐 나온 최종 데이터를 마지막으로 한 번 더 정규화(스케일링)해 줍니다.
        self.ln_f = nn.LayerNorm(emb_dim)

        # 마지막으로 128차원의 데이터를 단어장 크기(vocab_size)로 복원하여 각 글자가 나타날 확률 점수(Logits)를 출력합니다.
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device) # 0부터 T-1까지의 정수 배열 생성

        tok = self.token_embedding(x) # 단어의 의미적 임베딩 결과 (B, T, emb_dim)
        pos = self.position_embedding(pos)[None] # 위치 임베딩 결과 (1, T, emb_dim) -> 브로드캐스팅되어 B행만큼 확장됨

        # 통계학의 가법 모델(Additive Model)처럼, 단어 의미 벡터에 위치 벡터를 더해 최종 입력값을 만듭니다.
        h = tok + pos

        h = self.blocks(h) # 4개의 GPT 블록을 순차적으로 통과시킵니다.
        h = self.ln_f(h)   # 최종 레이어 정규화
        logits = self.lm_head(h) # 선형 변환을 거쳐 예측 점수 도출
        return logits

# 모델 인스턴스를 생성하고 임의의 입력 데이터(xb)를 주입해 봅니다.
model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape) # 출력 결과: (64, 64, 65) -> (Batch_size, Sequence_length, Vocab_size)

logits.shape: torch.Size([64, 64, 65])


## 4. 학습

In [5]:
"""## 4. 학습"""
# 이 부분은 수치 최적화(Optimization) 알고리즘을 사용해 최적의 회귀 계수(weights)를 찾는 과정과 상통합니다.

def sequence_cross_entropy(logits, targets):
    # 크로스 엔트로피 손실 함수: 다중 분류 분석에서 사용하는 비용 함수(Cost function)입니다.
    # 모델이 오답을 확신할수록 패널티(Loss)가 기하급수적으로 커집니다.
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train() # 모델을 '학습 모드'로 전환 (드롭아웃이나 레이어 정규화가 활성화됩니다)
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)

        optimizer.zero_grad() # 미분 계산을 위해 기울기(Gradient) 저장 공간을 깨끗이 비웁니다.
        loss.backward()       # 역전파(Backpropagation): 오차를 뒤에서부터 추적해가며 각 가중치별 편미분 값을 구합니다.
        optimizer.step()      # 경사하강법(Gradient Descent): 기울기 방향으로 가중치를 조금씩 업데이트합니다.

        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyGPT(vocab_size, block_size).to(device)

# AdamW 알고리즘: 경사하강법의 업그레이드 버전으로, 학습률(Learning Rate)을 데이터에 맞게 유동적으로 조절하며 찾아갑니다.
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# 100번 반복(Epoch)하여 최적의 가중치를 학습시킵니다.
for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.6649
epoch  1 | train loss 2.2737
epoch  2 | train loss 2.0932
epoch  3 | train loss 1.9735
epoch  4 | train loss 1.8859
epoch  5 | train loss 1.8177
epoch  6 | train loss 1.7637
epoch  7 | train loss 1.7261
epoch  8 | train loss 1.6929
epoch  9 | train loss 1.6642
epoch 10 | train loss 1.6453
epoch 11 | train loss 1.6215
epoch 12 | train loss 1.6052
epoch 13 | train loss 1.5876
epoch 14 | train loss 1.5755
epoch 15 | train loss 1.5627
epoch 16 | train loss 1.5485
epoch 17 | train loss 1.5414
epoch 18 | train loss 1.5341
epoch 19 | train loss 1.5194
epoch 20 | train loss 1.5156
epoch 21 | train loss 1.5087
epoch 22 | train loss 1.5001
epoch 23 | train loss 1.4940
epoch 24 | train loss 1.4879
epoch 25 | train loss 1.4825
epoch 26 | train loss 1.4784
epoch 27 | train loss 1.4716
epoch 28 | train loss 1.4649
epoch 29 | train loss 1.4608
epoch 30 | train loss 1.4578
epoch 31 | train loss 1.4542
epoch 32 | train loss 1.4452
epoch 33 | train loss 1.4467
epoch 34 | tra

## 5. Sampling

In [6]:
"""## 5. Sampling"""

@torch.no_grad() # 추론할 때는 미분(기울기 계산)이 필요 없으므로 메모리 절약을 위해 기능을 차단합니다.
def sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400):
    model.eval() # 모델을 '평가 모드'로 전환 (학습 때 사용한 드롭아웃 등의 기능을 꺼서 일관된 출력을 유도합니다)

    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1) # 한 칸씩 밀어가며 문맥 업데이트

    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :] # 문장의 맨 마지막 글자 자리에 온 예측 값(Logits)만 추출합니다.

        probs = F.softmax(logits, dim=-1) # 점수를 다중 분류 확률 값(0~1 사이, 총합 1)으로 바꿉니다.

        # [torch.multinomial]: 확률적 샘플링 기법입니다.
        # 가장 확률이 높은 글자만 고집하면 뻔하고 웅얼거리는 문장만 나오기 때문에,
        # 통계학의 누적 분포 함수 샘플링처럼 주어진 확률 분포에 따라 무작위로 글자를 추출합니다.
        ix = torch.multinomial(probs, num_samples=1)

        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

# 완벽한 문장은 아니더라도, 셰익스피어 희곡의 구조(인물 이름 뒤에 콜론, 줄바꿈 등)를 흉내 낸 문장이 출력됩니다.
print(sample_gpt(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=500))

ROMEO:
How fear'st dowry Henry and to do so of those
Befold them for this offer'd, from her unto the pointed
Throngs may thee in this he was securely and Lucio's
Son Countifely's head,
Your bawls in her and no form
To execute, that between your way;
Before enough she was some bays deal home to the wind
Lord Hastings, to lay an old one men and redeement,
I'll tell thee you 'ave for these and this false?

LUCIO:
Then, fast thou shalt unwas matter Pompey,
And sleep for auspice me in this.

CLIFFORD:
Vouc


## 6. 정리

이제 아주 작은 GPT 구조가 완성되었습니다.

전체 흐름은 다음과 같습니다.

1. bigram
2. MLP on names
3. MLP on Shakespeare
4. GPT-style dataset + minimal sequence model
5. single-head masked self-attention
6. tiny GPT